In [1]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import tiktoken

# 1. CONFIG

GPT2_SMALL_CONFIG = {
    "vocab_size": 50257,
    "context_length": 1024,
    "embed_dim": 768,
    "n_heads": 12,
    "n_layers": 12,
    "drop_rate": 0.1,
    "qkv_bias": True,
}


# 2. TOKENIZER (tiktoken)

def get_tokenizer():
    """Returns GPT-2 tokenizer via tiktoken."""
    return tiktoken.get_encoding("gpt2")


# 3. DATASET & DATALOADER

class TextDataset(Dataset):
    """Sliding window dataset that tokenizes raw text with tiktoken."""

    def __init__(self, text: str, tokenizer, max_length: int, stride: int):
        self.input_ids = []
        self.target_ids = []

        token_ids = tokenizer.encode(text, allowed_special={"<|endoftext|>"})

        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk  = token_ids[i          : i + max_length]
            target_chunk = token_ids[i + 1      : i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]


def create_dataloader(text: str, batch_size: int = 4, max_length: int = 256,
                      stride: int = 128, shuffle: bool = True,
                      drop_last: bool = True, num_workers: int = 0):
    tokenizer = get_tokenizer()
    dataset   = TextDataset(text, tokenizer, max_length, stride)
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle,
                      drop_last=drop_last, num_workers=num_workers)


# 4. MULTI-HEAD CAUSAL SELF-ATTENTION

class MultiHeadAttention(nn.Module):
    def __init__(self, embed_dim: int, n_heads: int, context_length: int,
                 dropout: float, qkv_bias: bool = False):
        super().__init__()
        assert embed_dim % n_heads == 0, "embed_dim must be divisible by n_heads"

        self.n_heads   = n_heads
        self.head_dim  = embed_dim // n_heads
        self.embed_dim = embed_dim

        self.W_qkv = nn.Linear(embed_dim, 3 * embed_dim, bias=qkv_bias)
        self.W_out = nn.Linear(embed_dim, embed_dim)
        self.attn_drop = nn.Dropout(dropout)
        self.resid_drop = nn.Dropout(dropout)

        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length, context_length), diagonal=1).bool()
        )

    def forward(self, x):
        B, T, C = x.shape

        qkv = self.W_qkv(x)                          # (B, T, 3C)
        q, k, v = qkv.split(self.embed_dim, dim=-1)  # each (B, T, C)

        def split_heads(t):
            return t.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)

        q, k, v = split_heads(q), split_heads(k), split_heads(v)

        # Scaled dot-product attention
        scale  = self.head_dim ** -0.5
        scores = (q @ k.transpose(-2, -1)) * scale    # (B, n_heads, T, T)
        scores = scores.masked_fill(self.mask[:T, :T], float("-inf"))
        weights = torch.softmax(scores, dim=-1)
        weights = self.attn_drop(weights)

        out = weights @ v                              # (B, n_heads, T, head_dim)
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        out = self.W_out(out)
        return self.resid_drop(out)


# 5. FEED-FORWARD NETWORK

class FeedForward(nn.Module):
    def __init__(self, embed_dim: int, dropout: float):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(embed_dim, 4 * embed_dim),
            nn.GELU(),
            nn.Linear(4 * embed_dim, embed_dim),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


# 6. TRANSFORMER BLOCK


class TransformerBlock(nn.Module):
    def __init__(self, cfg: dict):
        super().__init__()
        self.attn = MultiHeadAttention(
            embed_dim=cfg["embed_dim"],
            n_heads=cfg["n_heads"],
            context_length=cfg["context_length"],
            dropout=cfg["drop_rate"],
            qkv_bias=cfg["qkv_bias"],
        )
        self.ff   = FeedForward(cfg["embed_dim"], cfg["drop_rate"])
        self.ln1  = nn.LayerNorm(cfg["embed_dim"])
        self.ln2  = nn.LayerNorm(cfg["embed_dim"])

    def forward(self, x):
        x = x + self.attn(self.ln1(x))  # pre-LayerNorm
        x = x + self.ff(self.ln2(x))
        return x


# 7. GPT-2 MODEL

class GPTModel(nn.Module):
    def __init__(self, cfg: dict):
        super().__init__()
        self.tok_emb  = nn.Embedding(cfg["vocab_size"], cfg["embed_dim"])
        self.pos_emb  = nn.Embedding(cfg["context_length"], cfg["embed_dim"])
        self.drop_emb = nn.Dropout(cfg["drop_rate"])
        self.blocks   = nn.Sequential(
            *[TransformerBlock(cfg) for _ in range(cfg["n_layers"])]
        )
        self.ln_f     = nn.LayerNorm(cfg["embed_dim"])
        self.head     = nn.Linear(cfg["embed_dim"], cfg["vocab_size"], bias=False)


        self.head.weight = self.tok_emb.weight

        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx):
        B, T = idx.shape
        device = idx.device

        tok = self.tok_emb(idx)
        pos = self.pos_emb(torch.arange(T, device=device))
        x   = self.drop_emb(tok + pos)
        x   = self.blocks(x)
        x   = self.ln_f(x)
        return self.head(x)                            # (B, T, vocab_size)


# 8. TEXT GENERATION UTILITY

@torch.no_grad()
def generate(model, idx, max_new_tokens: int, context_size: int,
             temperature: float = 1.0, top_k: int = None):
    """Autoregressive token generation with temperature and top-k sampling."""
    model.eval()
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -context_size:]
        logits   = model(idx_cond)[:, -1, :]          # (B, vocab_size)

        if temperature != 1.0:
            logits = logits / temperature
        if top_k is not None:
            v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
            logits[logits < v[:, [-1]]] = float("-inf")

        probs    = torch.softmax(logits, dim=-1)
        idx_next = torch.multinomial(probs, num_samples=1)
        idx      = torch.cat([idx, idx_next], dim=1)
    return idx


def text_to_ids(text: str, tokenizer, device):
    return torch.tensor(tokenizer.encode(text)).unsqueeze(0).to(device)


def ids_to_text(ids: torch.Tensor, tokenizer):
    return tokenizer.decode(ids.squeeze(0).tolist())



# QUICK SMOKE-TEST


if __name__ == "__main__":
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Device: {device}")

    cfg   = GPT2_SMALL_CONFIG
    model = GPTModel(cfg).to(device)
    total = sum(p.numel() for p in model.parameters())
    print(f"Model parameters: {total:,}")

    tokenizer = get_tokenizer()
    sample    = "The transformer architecture revolutionized NLP"
    ids       = text_to_ids(sample, tokenizer, device)
    out       = model(ids)
    print(f"Forward pass output shape: {out.shape}")   # (1, T, 50257)

    generated = generate(model, ids, max_new_tokens=20,
                         context_size=cfg["context_length"],
                         temperature=0.8, top_k=40)
    print("Generated:", ids_to_text(generated, tokenizer))
    print("Stage 1 complete.")

Device: cpu
Model parameters: 124,439,808
Forward pass output shape: torch.Size([1, 7, 50257])
Generated: The transformer architecture revolutionized NLPbaPhonelements Struct StructPr revenge Bron 332 TRUE Rodriguez Indians Rodriguezazeera Rodriguez revenge merchandiseantom silhouette snapping
Stage 1 complete.
